
**Step 2 of 3 — Feature Enrichment**

Joins synthetic cardholder and merchant attributes to each cleaned transaction row, producing an enriched dataframe ready for MongoDB ingestion.

**Run order:** ingest.ipynb → enrich.ipynb → load_mongo.ipynb

In [ ]:
#Imports & Logging Setup
import logging
import numpy as np
import pandas as pd

# Configure logging to both file and notebook output
logging.basicConfig(
    filename="enrich.log",
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
logging.getLogger("").addHandler(console)

print("Imports complete.")

In [ ]:
#Configuration
INPUT_PATH = "cleaned.csv"
OUTPUT_PATH = "enriched.csv"
# Merchant Category Codes and regions for synthetic merchant assignment
MCC_POOL = ["5411", "5812", "4111", "5999", "7011", "5912", "4814", "5651"]
REGION_POOL = ["Western Europe", "Northern Europe", "Southern Europe", "Eastern Europe"]
# Seed for reproducibility
RNG_SEED = 42

In [ ]:
#Load Cleaned Data
def load_cleaned(path: str) -> pd.DataFrame:
    logging.info(f"Loading cleaned data from {path}")
    try:
        df = pd.read_csv(path)
        logging.info(f"Loaded {len(df)} rows.")
        return df
    except FileNotFoundError as e:
        logging.error(f"File not found: {e}")
        raise

df = load_cleaned(INPUT_PATH)
df.head()

In [ ]:
#Add Transaction IDs
def add_transaction_id(df: pd.DataFrame) -> pd.DataFrame:
    #Add a unique transaction_id to each row based on its index.
    logging.info("Adding transaction IDs...")
    df = df.reset_index(drop=True)
    df["transaction_id"] = [f"txn_{i:06d}" for i in df.index]
    return df
df = add_transaction_id(df)

In [ ]:
#Add synthetic cardholder sub-document fields
def add_cardholder_features(df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    logging.info("Adding synthetic cardholder features...")
    n = len(df)
    #cardholder_account_age_days : randomly sampled 30–3650 days
    df["cardholder_account_age_days"] = rng.integers(30, 3651, size=n)

    # Base avg monthly spend on transaction Amount with Gaussian noise
    #cardholder_avg_monthly_spend: estimated from Amount with added noise
    df["cardholder_avg_monthly_spend"] = (
        df["Amount"] * rng.uniform(0.8, 20.0, size=n)
    ).round(2).clip(lower=1.0)
    #cardholder_velocity_7d: randomly sampled 1–50 transactions
    df["cardholder_velocity_7d"] = rng.integers(1, 51, size=n)
    logging.info("Cardholder features added.")
    return df

rng = np.random.default_rng(RNG_SEED)
df = add_cardholder_features(df, rng)

In [ ]:
#Add Synthetic Merchant Features
def add_merchant_features(df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    logging.info("Adding synthetic merchant features...")
    n = len(df)
    #merchant_mcc --> randomly sampled from MCC_POOL
    df["merchant_mcc"] = rng.choice(MCC_POOL, size=n)
    #merchant_region --> randomly sampled from REGION_POOL
    df["merchant_region"] = rng.choice(REGION_POOL, size=n)
    #merchant_id --> deterministic string ID based on random integer
    df["merchant_id"] = [f"merch_{i:05d}" for i in rng.integers(1, 10001, size=n)]
    logging.info("Merchant features added.")
    return df

df = add_merchant_features(df, rng)

In [ ]:
#Save Enriched Data
df.to_csv(OUTPUT_PATH, index=False)